# Save best-fold models per language condition

Retrains the best CV fold for each condition (english / chinese / all) and saves
the head weights + config for later inference via `common.retrained_model`.

**Output:** `saved_models/<condition>/{head_weights.pt, config.json}`.

Training logic lives in `retrain_lib.py`; this notebook sets the best folds and
saves. Requires the repo (`common/` + `4_model_retraining/`) on the Colab path.

In [ ]:
!pip install -q soundfile librosa resampy transformers

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports & repo path

In [ ]:
import sys, json
from pathlib import Path
import torch

# Repo root on Colab: must contain common/ and 4_model_retraining/ (git clone or Drive sync).
REPO_ROOT = Path("/content/drive/MyDrive/ser_andreaingoglia_unimi")
sys.path.insert(0, str(REPO_ROOT / "4_model_retraining"))

import retrain_lib as R
from retrain_lib import TrainConfig

device = R.set_seed(42)
print(f"Device: {device}")

## 3. Config & best folds

In [ ]:
ESD_GT_CSV     = Path("/content/drive/MyDrive/ESD/ESD_GT_Colabo.csv")
ESD_ZIP_PATH   = Path("/content/drive/MyDrive/ESD_ZIP.zip")
ESD_AUDIO_ROOT = Path("/content/drive/MyDrive/ESD")
LOCAL_CACHE    = Path("/content/audio_cache_esd")
SAVE_ROOT      = Path("/content/drive/MyDrive/saved_models")

cfg = TrainConfig(num_epochs=15, dropout=0.1, balance_classes=True)

# Best CV folds found by head_retrain_language.ipynb
BEST_FOLDS = {
    "english": {"train_speakers": ["14", "13", "19", "16", "20", "15", "11", "12"],
                "val_speakers": ["17", "18"], "best_epoch": 7,  "best_acc": 0.8649},
    "chinese": {"train_speakers": ["7", "2", "8", "5", "6", "4", "1", "10"],
                "val_speakers": ["3", "9"],   "best_epoch": 10, "best_acc": 0.8151},
    "all":     {"train_speakers": ["16", "11", "13", "15", "12", "17", "14", "19",
                                   "8", "3", "1", "2", "10", "4", "5", "6"],
                "val_speakers": ["18", "20", "7", "9"], "best_epoch": 13, "best_acc": 0.7911},
}
for c, info in BEST_FOLDS.items():
    print(f"  {c}: train={len(info['train_speakers'])} spk | val={info['val_speakers']}")

## 4. Load ESD audio + ground truth

In [ ]:
cache = R.prepare_local_cache(ESD_ZIP_PATH, LOCAL_CACHE, ESD_AUDIO_ROOT)
path_map = R.build_path_map(cache)
esd_df = R.load_esd_gt(ESD_GT_CSV, path_map)

LABELS = R.SHARED_EMOTIONS
label2id = {lab: i for i, lab in enumerate(LABELS)}
print(f"Total samples: {len(esd_df)} | speakers: {sorted(esd_df['speaker'].unique())}")

## 5. Train & save each condition

In [ ]:
def train_and_save(condition, train_spk, val_spk):
    print(f"\n{'='*60}\n  CONDITION: {condition} | val={val_spk}\n{'='*60}")
    res = R.train_one(esd_df, train_spk, val_spk, LABELS, cfg, device)

    save_dir = SAVE_ROOT / condition
    save_dir.mkdir(parents=True, exist_ok=True)
    torch.save(res["best_state"], str(save_dir / "head_weights.pt"))

    config = {
        "condition": condition, "model_id": R.DEFAULT_MODEL,
        "embed_dim": res["embed_dim"], "num_labels": len(LABELS), "labels": LABELS,
        "label2id": label2id, "proj_dim": cfg.proj_dim, "hidden_dim": cfg.hidden_dim,
        "dropout": cfg.dropout, "input_layernorm": True,
        "train_speakers": train_spk, "val_speakers": val_spk,
        "best_epoch": res["best_epoch"], "best_val_acc": float(res["best_val_acc"]),
        "lr": cfg.lr, "weight_decay": cfg.weight_decay,
        "label_smoothing": cfg.label_smooth, "seed": cfg.seed,
    }
    with open(save_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    print(f"  Saved to {save_dir}: best_acc={res['best_val_acc']:.4f} @epoch {res['best_epoch']}")
    return res["best_val_acc"], res["best_epoch"]

## 6. Run all conditions

In [ ]:
results = {}
for cond, info in BEST_FOLDS.items():
    acc, ep = train_and_save(cond, info["train_speakers"], info["val_speakers"])
    results[cond] = {"acc": acc, "epoch": ep}

print(f"\n{'='*60}\n  SUMMARY\n{'='*60}")
for cond, r in results.items():
    print(f"  {cond:<10s}: acc={r['acc']:.4f} epoch={r['epoch']} -> {SAVE_ROOT / cond}")

## 7. Loading a saved model later

Use the shared loader in `common.retrained_model` — it rebuilds the head from
`config.json`, loads `head_weights.pt`, and pairs it with the frozen backbone:

```python
from common.retrained_model import load_retrained_model, embed_meanpool

bundle = load_retrained_model("saved_models/english", device="cpu")
extractor, backbone, head = bundle["extractor"], bundle["backbone"], bundle["head"]

inputs = extractor(wav, sampling_rate=16000, return_tensors="pt")
logits = head(embed_meanpool(backbone, inputs))
```